In [2]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load your synthetic UPI dataset
df = pd.read_csv('../data/raw/upi_synthetic.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

# Load accounts (needed for graph)
df_accounts = pd.read_csv('../data/raw/upi_accounts.csv')

print(f"✅ Transactions loaded : {df.shape}")
print(f"✅ Accounts loaded     : {df_accounts.shape}")
print(f"✅ Fraud rate          : {df['is_fraud'].mean()*100:.2f}%")
print(f"\nColumns: {df.columns.tolist()}")

✅ Transactions loaded : (50000, 9)
✅ Accounts loaded     : (5000, 6)
✅ Fraud rate          : 10.37%

Columns: ['txn_id', 'sender_id', 'sender_vpa', 'receiver_id', 'receiver_vpa', 'amount', 'timestamp', 'txn_type', 'is_fraud']


In [3]:
# Hour and time-based features
df['hour']        = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek   # 0=Monday, 6=Sunday
df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)
df['is_night']    = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)

# Amount features
df['log_amount']  = np.log1p(df['amount'])   # log scale smooths extreme values
df['near_limit']  = ((df['amount'] >= 95000) & 
                     (df['amount'] < 100000)).astype(int)

# Transaction type features — convert text to numbers (model needs numbers)
df['is_p2p']      = (df['txn_type'] == 'P2P').astype(int)
df['is_merchant'] = (df['txn_type'] == 'MERCHANT').astype(int)
df['is_bill']     = (df['txn_type'] == 'BILL').astype(int)

print("✅ Basic features added")
print(f"\nNew columns added:")
new_cols = ['hour','day_of_week','is_weekend','is_night',
            'log_amount','near_limit','is_p2p','is_merchant','is_bill']
for col in new_cols:
    print(f"  {col:20s} — sample values: {df[col].unique()[:5].tolist()}")

✅ Basic features added

New columns added:
  hour                 — sample values: [0, 1, 2, 4, 5]
  day_of_week          — sample values: [0, 1, 2, 3, 4]
  is_weekend           — sample values: [0, 1]
  is_night             — sample values: [1, 0]
  log_amount           — sample values: [5.735281141715572, 9.157162144322848, 5.279236478735244, 7.2653527972182985, 11.19422839595565]
  near_limit           — sample values: [0, 1]
  is_p2p               — sample values: [1, 0]
  is_merchant          — sample values: [0, 1]
  is_bill              — sample values: [0, 1]


In [4]:
# Sort by time first — critical for rolling windows to work correctly
df = df.sort_values('timestamp').reset_index(drop=True)

# Convert timestamp to seconds (makes window math easier)
df['ts_seconds'] = df['timestamp'].astype(np.int64) // 10**9

print("Computing velocity features — takes ~30 seconds...")

# For each transaction, count how many times the SENDER
# transacted in the last 1 hour and last 24 hours

velocity_1h  = []
velocity_24h = []

for i, row in df.iterrows():
    sender    = row['sender_id']
    t         = row['ts_seconds']
    
    # All rows where same sender AND within time window
    same_sender = df[df['sender_id'] == sender]['ts_seconds']
    
    v1h  = ((same_sender >= t - 3600)  & (same_sender <= t)).sum()
    v24h = ((same_sender >= t - 86400) & (same_sender <= t)).sum()
    
    velocity_1h.append(v1h)
    velocity_24h.append(v24h)
    
    # Progress update every 10000 rows
    if i % 10000 == 0:
        print(f"  Processed {i:,} / {len(df):,} rows...")

df['velocity_1h']  = velocity_1h
df['velocity_24h'] = velocity_24h

print(f"\n✅ Velocity features added")
print(f"Max velocity 1h  : {df['velocity_1h'].max()}  (high = suspicious)")
print(f"Max velocity 24h : {df['velocity_24h'].max()}")
print(f"\nVelocity stats for FRAUD transactions:")
print(df[df['is_fraud']==1][['velocity_1h','velocity_24h']].describe().round(2))
print(f"\nVelocity stats for LEGIT transactions:")
print(df[df['is_fraud']==0][['velocity_1h','velocity_24h']].describe().round(2))

Computing velocity features — takes ~30 seconds...
  Processed 0 / 50,000 rows...
  Processed 10,000 / 50,000 rows...
  Processed 20,000 / 50,000 rows...
  Processed 30,000 / 50,000 rows...
  Processed 40,000 / 50,000 rows...

✅ Velocity features added
Max velocity 1h  : 16  (high = suspicious)
Max velocity 24h : 16

Velocity stats for FRAUD transactions:
       velocity_1h  velocity_24h
count      5186.00       5186.00
mean          1.57          1.67
std           2.38          2.37
min           1.00          1.00
25%           1.00          1.00
50%           1.00          1.00
75%           1.00          1.00
max          16.00         16.00

Velocity stats for LEGIT transactions:
       velocity_1h  velocity_24h
count     44814.00      44814.00
mean          1.01          1.11
std           0.07          0.34
min           1.00          1.00
25%           1.00          1.00
50%           1.00          1.00
75%           1.00          1.00
max           3.00          5.00


In [5]:
print("Building graph with mule-aware structure...")

G = nx.DiGraph()

# Add all accounts as nodes with mule attribute
for _, acc in df_accounts.iterrows():
    G.add_node(acc['account_id'], is_mule=acc['is_mule'])

print(f"✅ Nodes added : {G.number_of_nodes()}")

# Add transactions as edges
for _, row in df.iterrows():
    s = row['sender_id']
    r = row['receiver_id']
    w = row['amount']
    if G.has_edge(s, r):
        G[s][r]['weight'] += w
        G[s][r]['count']  += 1
    else:
        G.add_edge(s, r, weight=w, count=1)

print(f"✅ Edges added : {G.number_of_edges()}")
print()

# Compute graph features
pagerank   = nx.pagerank(G, weight='weight')
in_degree  = dict(G.in_degree())
out_degree = dict(G.out_degree())

# Map to transactions
df['sender_pagerank']    = df['sender_id'].map(pagerank).fillna(0)
df['sender_in_degree']   = df['sender_id'].map(in_degree).fillna(0)
df['sender_out_degree']  = df['sender_id'].map(out_degree).fillna(0)
df['receiver_pagerank']  = df['receiver_id'].map(pagerank).fillna(0)
df['receiver_in_degree'] = df['receiver_id'].map(in_degree).fillna(0)

print("✅ Graph features merged")
print()

# Verify mule accounts now dominate PageRank
print("=== TOP 10 ACCOUNTS BY PAGERANK ===")
top_pagerank = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:10]
mule_count = 0
for acc, score in top_pagerank:
    is_mule = df_accounts[df_accounts['account_id']==acc]['is_mule'].values[0]
    label   = '🔴 MULE' if is_mule else '✅ Normal'
    if is_mule:
        mule_count += 1
    print(f"  {acc} | PageRank: {score:.6f} | {label}")

print()
print(f"Mules in top 10: {mule_count}/10")
print()

# Check feature separation
graph_cols = ['sender_pagerank','sender_in_degree',
              'sender_out_degree','receiver_pagerank',
              'receiver_in_degree']
print("=== GRAPH FEATURES: FRAUD vs LEGIT ===")
print(df.groupby('is_fraud')[graph_cols].mean().round(6))

Building graph with mule-aware structure...
✅ Nodes added : 5000
✅ Edges added : 49949

✅ Graph features merged

=== TOP 10 ACCOUNTS BY PAGERANK ===
  ACC00425 | PageRank: 0.000925 | 🔴 MULE
  ACC00141 | PageRank: 0.000789 | 🔴 MULE
  ACC00202 | PageRank: 0.000788 | 🔴 MULE
  ACC00197 | PageRank: 0.000771 | 🔴 MULE
  ACC00193 | PageRank: 0.000743 | 🔴 MULE
  ACC01168 | PageRank: 0.000731 | ✅ Normal
  ACC00264 | PageRank: 0.000716 | 🔴 MULE
  ACC00381 | PageRank: 0.000712 | 🔴 MULE
  ACC03331 | PageRank: 0.000709 | ✅ Normal
  ACC00353 | PageRank: 0.000686 | 🔴 MULE

Mules in top 10: 8/10

=== GRAPH FEATURES: FRAUD vs LEGIT ===
          sender_pagerank  sender_in_degree  sender_out_degree  \
is_fraud                                                         
0                0.000197          9.811800          10.954345   
1                0.000221         11.247975          10.891631   

          receiver_pagerank  receiver_in_degree  
is_fraud                                         
0        

In [6]:
print("Computing graph features — takes about 30 seconds...")

# PageRank — measures how 'important' each node is in the network
# Mule accounts score high because many accounts send money TO them
pagerank = nx.pagerank(G, weight='weight')

# In-degree — how many UNIQUE accounts sent money TO this account
# Mules have very high in-degree (many senders, one receiver)
in_degree = dict(G.in_degree())

# Out-degree — how many UNIQUE accounts this account sent money TO
# Normal for most, high for accounts spreading money outward
out_degree = dict(G.out_degree())

print("✅ PageRank computed")
print("✅ Degree features computed")

# Map features back to each TRANSACTION using sender and receiver IDs
df['sender_pagerank']    = df['sender_id'].map(pagerank).fillna(0)
df['sender_in_degree']   = df['sender_id'].map(in_degree).fillna(0)
df['sender_out_degree']  = df['sender_id'].map(out_degree).fillna(0)
df['receiver_pagerank']  = df['receiver_id'].map(pagerank).fillna(0)
df['receiver_in_degree'] = df['receiver_id'].map(in_degree).fillna(0)

print("✅ Graph features merged into dataframe")
print()

# Verify — mule accounts should have higher PageRank
print("=== TOP 5 ACCOUNTS BY PAGERANK ===")
top_pagerank = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:5]
for acc, score in top_pagerank:
    is_mule = df_accounts[df_accounts['account_id']==acc]['is_mule'].values[0]
    label = '🔴 MULE' if is_mule else '✅ Normal'
    print(f"  {acc} | PageRank: {score:.6f} | {label}")

print()
print("=== GRAPH FEATURES: FRAUD vs LEGIT ===")
graph_cols = ['sender_pagerank','sender_in_degree',
              'sender_out_degree','receiver_in_degree']
print(df.groupby('is_fraud')[graph_cols].mean().round(4))

Computing graph features — takes about 30 seconds...
✅ PageRank computed
✅ Degree features computed
✅ Graph features merged into dataframe

=== TOP 5 ACCOUNTS BY PAGERANK ===
  ACC00425 | PageRank: 0.000925 | 🔴 MULE
  ACC00141 | PageRank: 0.000789 | 🔴 MULE
  ACC00202 | PageRank: 0.000788 | 🔴 MULE
  ACC00197 | PageRank: 0.000771 | 🔴 MULE
  ACC00193 | PageRank: 0.000743 | 🔴 MULE

=== GRAPH FEATURES: FRAUD vs LEGIT ===
          sender_pagerank  sender_in_degree  sender_out_degree  \
is_fraud                                                         
0                  0.0002            9.8118            10.9543   
1                  0.0002           11.2480            10.8916   

          receiver_in_degree  
is_fraud                      
0                    10.9761  
1                    16.6057  


In [7]:
feature_cols = [
    # Time features
    'hour', 'day_of_week', 'is_weekend', 'is_night',
    # Amount features
    'log_amount', 'near_limit',
    # Transaction type
    'is_p2p', 'is_merchant', 'is_bill',
    # Velocity features
    'velocity_1h', 'velocity_24h',
    # Graph features
    'sender_pagerank', 'sender_in_degree',
    'sender_out_degree', 'receiver_pagerank',
    'receiver_in_degree',
    # Label
    'is_fraud'
]

df_features = df[feature_cols].dropna()

import os
os.makedirs('../data/processed', exist_ok=True)
df_features.to_csv('../data/processed/features.csv', index=False)

print(f"✅ Feature matrix saved")
print(f"Shape      : {df_features.shape}")
print(f"Features   : {len(feature_cols)-1} input features + 1 label")
print(f"Fraud rate : {df_features['is_fraud'].mean()*100:.2f}%")
print()
print("=== FINAL FEATURE LIST ===")
for i, col in enumerate([c for c in feature_cols if c != 'is_fraud'], 1):
    print(f"  {i:02d}. {col}")
print()


✅ Feature matrix saved
Shape      : (50000, 17)
Features   : 16 input features + 1 label
Fraud rate : 10.37%

=== FINAL FEATURE LIST ===
  01. hour
  02. day_of_week
  03. is_weekend
  04. is_night
  05. log_amount
  06. near_limit
  07. is_p2p
  08. is_merchant
  09. is_bill
  10. velocity_1h
  11. velocity_24h
  12. sender_pagerank
  13. sender_in_degree
  14. sender_out_degree
  15. receiver_pagerank
  16. receiver_in_degree



In [8]:
# ── Build Account Feature Store ───────────────────────────────────────────────
# This simulates what a real feature store does
# Pre-compute all account-level features so the API can look them up

# Velocity per account — average over all their transactions
velocity_store = df.groupby('sender_id').agg(
    avg_velocity_1h  = ('velocity_1h',  'mean'),
    avg_velocity_24h = ('velocity_24h', 'mean'),
    max_velocity_1h  = ('velocity_1h',  'max'),
    txn_count        = ('txn_id',       'count'),
    avg_amount       = ('amount',       'mean'),
).round(4)

# Graph features per account (already computed)
graph_store = pd.DataFrame({
    'account_id':    list(pagerank.keys()),
    'pagerank':      list(pagerank.values()),
    'in_degree':     [in_degree.get(a, 0) for a in pagerank.keys()],
    'out_degree':    [out_degree.get(a, 0) for a in pagerank.keys()],
})

# Merge into one feature store
feature_store = graph_store.merge(
    velocity_store,
    left_on='account_id',
    right_index=True,
    how='left'
).fillna(0)

# Add mule flag from accounts
feature_store = feature_store.merge(
    df_accounts[['account_id','is_mule','vpa']],
    on='account_id',
    how='left'
)

feature_store = feature_store.round(6)

# Save
feature_store.to_csv('../data/processed/account_feature_store.csv', index=False)

print(f"✅ Account feature store saved")
print(f"Shape    : {feature_store.shape}")
print(f"Accounts : {len(feature_store):,}")
print()
print("Sample — top 5 mule accounts by PageRank:")
print(
    feature_store[feature_store['is_mule']==1]
    .sort_values('pagerank', ascending=False)
    .head(5)[['account_id','vpa','pagerank',
               'in_degree','avg_velocity_1h']]
    .to_string()
)

✅ Account feature store saved
Shape    : (5000, 11)
Accounts : 5,000

Sample — top 5 mule accounts by PageRank:
    account_id                               vpa  pagerank  in_degree  avg_velocity_1h
425   ACC00425             ishanvi.bahl262@paytm  0.000925         18              1.0
141   ACC00141              hitesh.bava767@oksbi  0.000789         23              1.0
202   ACC00202             maanas.chhabra439@ybl  0.000788         28              1.0
197   ACC00197              maanas.kannan694@upi  0.000771         26              1.0
193   ACC00193  warinder.nagarajan485@okhdfcbank  0.000743         15              1.0
